# Chinese Understanding Benchmark: Qwen3-4B-Instruct-2507 vs Gemma 4 E4B-it

This notebook evaluates and optionally LoRA-finetunes two instruction-tuned 4B-class models on CLUE Chinese understanding tasks.

Default models:

- `Qwen/Qwen3-4B-Instruct-2507`
- `google/gemma-4-E4B-it`

The evaluation uses Chinese label names plus robust output parsing to avoid invalid predictions caused by free-form generation.
> Note: `Qwen/Qwen3.5-4B-Instruct` was removed from this notebook because that Hugging Face ID is not valid. The closest current 4B instruction-tuned Qwen checkpoint is `Qwen/Qwen3-4B-Instruct-2507`.


In [ ]:
# Optional install cell. Run only if your environment is missing packages.
# If you hit torchvision::nms errors, uninstall torchvision because this text benchmark does not need it.

# !pip install -U torch transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece
# !pip uninstall -y torchvision

In [1]:
import os
import re
import time
import gc
from typing import Dict, List, Any, Optional

import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from transformers import TrainingArguments
    from peft import LoraConfig, PeftModel
    from trl import SFTTrainer
    PEFT_AVAILABLE = True
except Exception as e:
    PEFT_AVAILABLE = False
    PEFT_IMPORT_ERROR = repr(e)

print('torch:', torch.__version__)
import transformers
print('transformers:', transformers.__version__)
print('peft/trl available:', PEFT_AVAILABLE)
if not PEFT_AVAILABLE:
    print('PEFT import error:', PEFT_IMPORT_ERROR)

torch: 2.12.0
transformers: 5.8.1
peft/trl available: True


In [2]:
# Main configuration

MODELS = {
    'qwen3_4b_instruct_2507': 'Qwen/Qwen3-4B-Instruct-2507',
    'gemma_e4b_it': 'google/gemma-4-E4B-it',
}

TASKS = ['afqmc', 'tnews', 'cmnli']
SPLIT = 'validation'
MAX_SAMPLES = 200
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# For Mac/CPU/MPS setups, use auto device placement. For CUDA machines, this also works.
DEVICE_MAP = 'auto'
TORCH_DTYPE = torch.float16

In [3]:
# Label specifications.
# We score dataset IDs, but ask models to output Chinese label names.

TASK_SPECS = {
    'afqmc': {
        'label_id_to_name': {0: '不同', 1: '相同'},
        'label_name_to_id': {'不同': '0', '相同': '1'},
    },
    'cmnli': {
        # Common Hugging Face CLUE CMNLI mapping: 0=entailment, 1=neutral, 2=contradiction.
        'label_id_to_name': {0: '蕴含', 1: '中立', 2: '矛盾'},
        'label_name_to_id': {'蕴含': '0', '中立': '1', '矛盾': '2'},
    },
    'tnews': {
        'label_id_to_name': {
            100: '故事', 101: '文化', 102: '娱乐', 103: '体育', 104: '财经',
            106: '房产', 107: '汽车', 108: '教育', 109: '科技', 110: '军事',
            112: '旅游', 113: '国际', 114: '股票', 115: '农业', 116: '电竞',
        },
        'label_name_to_id': {
            '故事': '100', '文化': '101', '娱乐': '102', '体育': '103', '财经': '104',
            '房产': '106', '汽车': '107', '教育': '108', '科技': '109', '军事': '110',
            '旅游': '112', '国际': '113', '股票': '114', '农业': '115', '电竞': '116',
        },
    },
}

LABEL_ALIASES = {
    'afqmc': {
        '相同': ['相同', '一致', '等价', '同义', '语义相同', '是', 'yes', 'true', '1'],
        '不同': ['不同', '不相同', '不一致', '不等价', '语义不同', '否', 'no', 'false', '0'],
    },
    'cmnli': {
        '蕴含': ['蕴含', '包含', '推出', 'entailment', 'entails', '0'],
        '中立': ['中立', '无关', '无法判断', 'neutral', '1'],
        '矛盾': ['矛盾', '冲突', 'contradiction', 'contradict', '2'],
    },
    'tnews': {
        '故事': ['故事', '100'], '文化': ['文化', '101'], '娱乐': ['娱乐', '102'],
        '体育': ['体育', '103'], '财经': ['财经', '104'], '房产': ['房产', '106'],
        '汽车': ['汽车', '107'], '教育': ['教育', '108'], '科技': ['科技', '109'],
        '军事': ['军事', '110'], '旅游': ['旅游', '112'], '国际': ['国际', '113'],
        '股票': ['股票', '114'], '农业': ['农业', '115'], '电竞': ['电竞', '116'],
    },
}

def get_label_names(task: str) -> List[str]:
    return list(TASK_SPECS[task]['label_name_to_id'].keys())

def label_id_to_name(task: str, label_id: Any) -> str:
    return TASK_SPECS[task]['label_id_to_name'].get(int(label_id), str(label_id))

def label_name_to_id(task: str, label_name: str) -> str:
    return TASK_SPECS[task]['label_name_to_id'].get(label_name, '__invalid__')

In [4]:
def build_prompt(task: str, example: Dict[str, Any]) -> str:
    labels = '、'.join(get_label_names(task))

    if task == 'afqmc':
        return f'''你是中文二分类器。只输出“相同”或“不同”其中一个词，不要解释。

句子1：{example['sentence1']}
句子2：{example['sentence2']}

语义是否相同？答案：'''

    if task == 'cmnli':
        return f'''你是中文自然语言推理分类器。只能输出一个标签，不要解释。

任务：判断“假设”与“前提”的关系。
可选标签：{labels}

前提：{example['sentence1']}
假设：{example['sentence2']}

答案：'''

    if task == 'tnews':
        return f'''你是中文新闻标题分类器。只能输出一个标签，不要解释。

任务：判断新闻标题所属类别。
可选标签：{labels}

标题：{example['sentence']}

答案：'''

    raise ValueError(f'Unsupported task: {task}')


def normalize_output(text: Any) -> str:
    text = str(text).strip().lower()
    for prefix in ['答案：', '答案:', '标签：', '标签:', '类别：', '类别:']:
        text = text.replace(prefix, '')
    text = text.replace('\n', ' ')
    text = re.sub(r'''[。，“”，、；;:：\[\]\(\)（）"']''', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_label_name(task: str, text: Any) -> str:
    text_norm = normalize_output(text)

    # 1. Exact alias match.
    for canonical_label, aliases in LABEL_ALIASES[task].items():
        for alias in aliases:
            if text_norm == normalize_output(alias):
                return canonical_label

    # 2. Substring alias match.
    for canonical_label, aliases in LABEL_ALIASES[task].items():
        for alias in aliases:
            alias_norm = normalize_output(alias)
            if alias_norm and alias_norm in text_norm:
                return canonical_label

    # 3. First standalone numeric label.
    digit_match = re.search(r'\b\d+\b', text_norm)
    if digit_match:
        digit = digit_match.group(0)
        for canonical_label, aliases in LABEL_ALIASES[task].items():
            if digit in aliases:
                return canonical_label

    return '__invalid__'


def format_example_for_sft(task: str, ex: Dict[str, Any]) -> str:
    return build_prompt(task, ex) + label_id_to_name(task, ex['label'])

In [5]:
def load_model_and_tokenizer(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


@torch.no_grad()
def generate_answer(tokenizer, model, prompt: str, max_new_tokens: int = 6) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()


def cleanup_model(model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [6]:
def evaluate_model_on_task(
    model_key: str,
    model_id: str,
    task: str,
    split: str = 'validation',
    max_samples: Optional[int] = 200,
    debug_first_n: int = 5,
):
    dataset = load_dataset('clue', task, split=split)
    if max_samples is not None:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    tokenizer, model = load_model_and_tokenizer(model_id)
    rows = []
    y_true, y_pred = [], []
    start = time.time()

    for i, ex in enumerate(tqdm(dataset, desc=f'{model_key}/{task}')):
        prompt = build_prompt(task, ex)
        raw = generate_answer(tokenizer, model, prompt, max_new_tokens=6)
        pred_name = extract_label_name(task, raw)
        pred_id = label_name_to_id(task, pred_name)
        gold_id = str(ex['label'])
        gold_name = label_id_to_name(task, gold_id)

        y_true.append(gold_id)
        y_pred.append(pred_id)
        rows.append({
            'model_key': model_key,
            'model_id': model_id,
            'task': task,
            'gold_id': gold_id,
            'gold_name': gold_name,
            'pred_id': pred_id,
            'pred_name': pred_name,
            'raw_output': raw,
        })

        if i < debug_first_n:
            print({'gold_id': gold_id, 'gold_name': gold_name, 'raw': repr(raw), 'pred_name': pred_name, 'pred_id': pred_id})

    seconds = time.time() - start
    summary = {
        'model_key': model_key,
        'model_id': model_id,
        'task': task,
        'split': split,
        'samples': len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'invalid_rate': sum(p == '__invalid__' for p in y_pred) / len(y_pred),
        'seconds': seconds,
        'samples_per_second': len(y_true) / seconds if seconds > 0 else None,
    }
    cleanup_model(model, tokenizer)
    return summary, pd.DataFrame(rows)

In [7]:
# Run baseline evaluation.
# Start with AFQMC to verify parsing; then run all tasks.

all_summaries = []
all_predictions = []

for model_key, model_id in MODELS.items():
    for task in TASKS:
        summary, pred_df = evaluate_model_on_task(
            model_key=model_key,
            model_id=model_id,
            task=task,
            split=SPLIT,
            max_samples=MAX_SAMPLES,
            debug_first_n=3,
        )
        print(summary)
        all_summaries.append(summary)
        all_predictions.append(pred_df)

summary_df = pd.DataFrame(all_summaries)
predictions_df = pd.concat(all_predictions, ignore_index=True)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'baseline_summary.csv'), index=False)
predictions_df.to_csv(os.path.join(RESULTS_DIR, 'baseline_predictions.csv'), index=False)
summary_df

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


gemma_e4b_it/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'gold_id': '0', 'gold_name': '不同', 'raw': "'：：：\\n\\n语义是否'", 'pred_name': '相同', 'pred_id': '1'}
{'gold_id': '0', 'gold_name': '不同', 'raw': "'：\\n\\n句子是否相同？'", 'pred_name': '相同', 'pred_id': '1'}
{'gold_id': '1', 'gold_name': '相同', 'raw': "'：：：\\n\\n语义是否'", 'pred_name': '相同', 'pred_id': '1'}


KeyboardInterrupt: 

In [ ]:
# Inspect invalid examples, if any.
invalids = predictions_df[predictions_df['pred_id'] == '__invalid__']
print('Invalid count:', len(invalids))
invalids.head(20)

## Optional: LoRA fine-tuning

This is optional and may be slow on a 32GB Mac. Start with one task, such as AFQMC. If memory is tight, lower `max_train_samples`, use batch size 1, and keep `gradient_accumulation_steps` modest.

In [ ]:
def finetune_lora_on_task(
    model_key: str,
    model_id: str,
    task: str = 'afqmc',
    output_dir: Optional[str] = None,
    max_train_samples: int = 1000,
    num_train_epochs: float = 1.0,
):
    if not PEFT_AVAILABLE:
        raise RuntimeError(f'PEFT/TRL are unavailable: {PEFT_IMPORT_ERROR}')

    if output_dir is None:
        output_dir = f'adapters/{model_key}_{task}_lora'

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )

    ds = load_dataset('clue', task, split='train')
    ds = ds.select(range(min(max_train_samples, len(ds))))
    ds = ds.map(lambda ex: {'text': format_example_for_sft(task, ex)})

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        task_type='CAUSAL_LM',
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=num_train_epochs,
        logging_steps=10,
        save_steps=250,
        save_total_limit=2,
        report_to='none',
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field='text',
        peft_config=lora_config,
        args=training_args,
        max_seq_length=512,
    )
    trainer.train()
    trainer.save_model(output_dir)
    cleanup_model(model, tokenizer)
    return output_dir

In [ ]:
# Example fine-tuning run. Uncomment one line at a time.

# qwen_adapter = finetune_lora_on_task('qwen3_4b_instruct_2507', MODELS['qwen3_4b_instruct_2507'], task='afqmc', max_train_samples=1000)
# gemma_adapter = finetune_lora_on_task('gemma_e4b_it', MODELS['gemma_e4b_it'], task='afqmc', max_train_samples=1000)

In [ ]:
def evaluate_lora_adapter_on_task(
    base_model_id: str,
    adapter_dir: str,
    model_key: str,
    task: str,
    split: str = 'validation',
    max_samples: Optional[int] = 200,
):
    if not PEFT_AVAILABLE:
        raise RuntimeError(f'PEFT is unavailable: {PEFT_IMPORT_ERROR}')

    dataset = load_dataset('clue', task, split=split)
    if max_samples is not None:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=TORCH_DTYPE, device_map=DEVICE_MAP, trust_remote_code=True)
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    rows, y_true, y_pred = [], [], []
    start = time.time()
    for ex in tqdm(dataset, desc=f'{model_key}/{task}/adapter'):
        prompt = build_prompt(task, ex)
        raw = generate_answer(tokenizer, model, prompt, max_new_tokens=6)
        pred_name = extract_label_name(task, raw)
        pred_id = label_name_to_id(task, pred_name)
        gold_id = str(ex['label'])
        gold_name = label_id_to_name(task, gold_id)
        y_true.append(gold_id)
        y_pred.append(pred_id)
        rows.append({'model_key': model_key, 'base_model_id': base_model_id, 'adapter_dir': adapter_dir, 'task': task, 'gold_id': gold_id, 'gold_name': gold_name, 'pred_id': pred_id, 'pred_name': pred_name, 'raw_output': raw})

    seconds = time.time() - start
    summary = {
        'model_key': model_key,
        'base_model_id': base_model_id,
        'adapter_dir': adapter_dir,
        'task': task,
        'split': split,
        'samples': len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'invalid_rate': sum(p == '__invalid__' for p in y_pred) / len(y_pred),
        'seconds': seconds,
        'samples_per_second': len(y_true) / seconds if seconds > 0 else None,
    }
    cleanup_model(model, tokenizer)
    return summary, pd.DataFrame(rows)

In [ ]:
# Example adapter evaluation. Uncomment after training.

# summary, preds = evaluate_lora_adapter_on_task(MODELS['qwen3_4b_instruct_2507'], qwen_adapter, 'qwen3_4b_instruct_2507_lora', 'afqmc')
# print(summary)
# preds.head()